## Evaluate on CVRPLib

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

from models.zoo.resym.model import ReSym

from envs.mtvrp.env import MTVRPEnv
from envs.symmtvrp.env import SymMTVRPEnv
from envs.mtvrp.generator import MTVRPGenerator

# Create env
generator = MTVRPGenerator(num_loc=100, variant_preset="all")

# Choose your model
env = MTVRPEnv(generator, check_solution=False)
# PATH = "pretrained/mtvrp100/te-am-None-None[]-multistart_sampling-False-0.0/299_-16.8492_-16.4223.ckpt"
# model = ReSym.load_from_checkpoint(PATH, map_location="cpu")
# result_name = "rf"

env = SymMTVRPEnv(generator, check_solution=False)
PATH = "pretrained/mtvrp100/te-bd/299_-16.8572_-16.4135.ckpt"
model = ReSym.load_from_checkpoint(PATH, map_location="cpu")
result_name = "rf-bd"

# PATH = "pretrained/mtvrp100/te-vam-None-None[]-multistart_sampling-True-0.0/299_-16.7821_-16.3895.ckpt"
# model = ReSym.load_from_checkpoint(PATH, map_location="cpu")
# result_name = "rf-vam"

policy = model.policy

/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
policy = policy.to(device).eval()

td_test = env.reset(env.generator(32))

# Test the model
with torch.amp.autocast("cuda"):
    with torch.inference_mode():
        out = policy(td_test.clone().to(device), env, phase="test", decode_type="greedy", return_actions=True)
        actions = out['actions'].cpu().detach()
        rewards = out['reward'].cpu().detach()

print(f"Average reward: {rewards.mean().item():.3f}")

/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/torch/nn/functional.py:2920: UserWarning: Mismatch dtype between input and weight: input dtype = c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


Average reward: -20.416


### VRPLib Evaluation

Note: run 

```
python routefinder/data/download_vrplib.py
```
 to download the VRPLib instances (then move them under this dir)

In [4]:
import os

# Ensure you have downloaded vrplib under vrplib/
# Initialize the instances dictionary
instances = {}

# Walk through the vrplib directory recursively
for root, dirs, files in sorted(os.walk('./vrplib')):
    for file in files:
        if file.endswith('.vrp'):
            # Initialize the dictionary for this instance
            instance_name = file[:-4]  # Remove the '.vrp' extension
            instances[instance_name] = {"solution": None}  # Create entry for instance
            # Print the file for verification
            instances[instance_name]["data"] = os.path.join(root, file)  # Save the VRP file path
            instances[instance_name]["solution"] = os.path.join(root, file[:-4] + '.sol')  # Save the solution file path
            # ensure the solution file exists
            assert os.path.exists(instances[instance_name]["solution"]), f"Solution file not found for {instance_name}"
            print(f"Found VRP file: {file}")

Found VRP file: A-n45-k6.vrp
Found VRP file: A-n45-k7.vrp
Found VRP file: A-n34-k5.vrp
Found VRP file: A-n38-k5.vrp
Found VRP file: A-n64-k9.vrp
Found VRP file: A-n36-k5.vrp
Found VRP file: A-n32-k5.vrp
Found VRP file: A-n62-k8.vrp
Found VRP file: A-n53-k7.vrp
Found VRP file: A-n63-k10.vrp
Found VRP file: A-n60-k9.vrp
Found VRP file: A-n65-k9.vrp
Found VRP file: A-n69-k9.vrp
Found VRP file: A-n39-k5.vrp
Found VRP file: A-n48-k7.vrp
Found VRP file: A-n39-k6.vrp
Found VRP file: A-n44-k6.vrp
Found VRP file: A-n54-k7.vrp
Found VRP file: A-n46-k7.vrp
Found VRP file: A-n37-k6.vrp
Found VRP file: A-n37-k5.vrp
Found VRP file: A-n33-k6.vrp
Found VRP file: A-n33-k5.vrp
Found VRP file: A-n55-k9.vrp
Found VRP file: A-n63-k9.vrp
Found VRP file: A-n61-k9.vrp
Found VRP file: A-n80-k10.vrp
Found VRP file: B-n31-k5.vrp
Found VRP file: B-n57-k9.vrp
Found VRP file: B-n63-k10.vrp
Found VRP file: B-n52-k7.vrp
Found VRP file: B-n50-k7.vrp
Found VRP file: B-n39-k5.vrp
Found VRP file: B-n78-k10.vrp
Found VRP 

In [5]:
from rl4co.utils.ops import unbatchify, gather_by_index

# Utils function
def normalize_coord(coord:torch.Tensor) -> torch.Tensor: # if we scale x and y separately, aren't we losing the relative position of the points? i.e. we mess with the distances.
    x, y = coord[:, 0], coord[:, 1]
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    x_scaled = (x - x_min) / (x_max - x_min) 
    y_scaled = (y - y_min) / (y_max - y_min)
    coord_scaled = torch.stack([x_scaled, y_scaled], dim=1)
    return coord_scaled 


def evaluate(model, td, env,
             num_augment=8,
             num_starts=None,
             ):
    
    with torch.inference_mode():
        with torch.amp.autocast("cuda"):
            n_start = env.get_num_starts(td) if num_starts is None else num_starts

            if num_augment > 1:
                td = model.augment(td)

            # Evaluate policy
            out = model.policy(
                td, env, phase="test", num_starts=n_start, return_actions=True
            )

            # Unbatchify reward to [batch_size, num_augment, num_starts].
            reward = unbatchify(out["reward"], (num_augment, n_start))

            if n_start > 1:
                # max multi-start reward
                max_reward, max_idxs = reward.max(dim=-1)
                out.update({"max_reward": max_reward})

                if out.get("actions", None) is not None:
                    # Reshape batch to [batch_size, num_augment, num_starts, ...]
                    actions = unbatchify(out["actions"], (num_augment, n_start))
                    out.update(
                        {"best_multistart_actions": gather_by_index(actions, max_idxs, dim=max_idxs.dim())}
                    )
                    out["actions"] = actions

            # Get augmentation score only during inference
            if num_augment > 1:
                # If multistart is enabled, we use the best multistart rewards
                reward_ = max_reward if n_start > 1 else reward
                max_aug_reward, max_idxs = reward_.max(dim=1)
                out.update({"max_aug_reward": max_aug_reward})

                if out.get("actions", None) is not None:
                    actions_ = (
                        out["best_multistart_actions"] if n_start > 1 else out["actions"]
                    )
                    out.update({"best_aug_actions": gather_by_index(actions_, max_idxs)})
                    
            return out

In [6]:
from tensordict import TensorDict
from math import ceil
import time
import vrplib


model.eval().to(device)

results = []

for instance in instances:
    problem = vrplib.read_instance(instances[instance]["data"])

    if problem.get("node_coord", None) is None:
        print(f"Skipping {instance} as it does not have node_coord")
        continue
    coords = torch.tensor(problem['node_coord']).float()
    coords_norm = normalize_coord(coords)

    original_capacity = problem['capacity']
    demand = torch.tensor(problem['demand'][1:]).float() / original_capacity
    original_capacity = torch.tensor(original_capacity)[None]         

    # Make instance
    td_instance = TensorDict({
        "locs": coords_norm,
        "demand_linehaul": demand,
        "capacity_original": original_capacity,
    },
    batch_size=[])[None]

    td_reset = env.reset(td_instance).to(device)
    
    start = time.time()
    actions = evaluate(model, td_reset.clone(), env)["best_aug_actions"]
    inference_time = time.time() - start        

    # Obtain reward from the environment with new locs
    td_reset["locs"] = coords[None] # unnormalized
    reward = env.get_reward(td_reset, actions)
    
    # Load the optimal cost
    solution = vrplib.read_solution(instances[instance]["solution"])
    optimal_cost = solution['cost'] # note that this cost is somehow slightly lower than the one calculated from the distance matrix
    
    # Calculate the gap and print
    cost = ceil(-reward.item())
    gap = (cost - optimal_cost) / optimal_cost
    print(f'Problem: {instance:<15} Cost: {cost:<10} Optimal Cost: {optimal_cost:<10}\t Gap: {gap:.3%}')
    
    results.append({
        "instance": instance,
        "cost": cost,
        "optimal_cost": optimal_cost,
        "gap": gap,
        "inference_time": inference_time,
    })

Problem: A-n45-k6        Cost: 985        Optimal Cost: 944       	 Gap: 4.343%
Problem: A-n45-k7        Cost: 1160       Optimal Cost: 1146      	 Gap: 1.222%
Problem: A-n34-k5        Cost: 788        Optimal Cost: 778       	 Gap: 1.285%
Problem: A-n38-k5        Cost: 751        Optimal Cost: 730       	 Gap: 2.877%
Problem: A-n64-k9        Cost: 1436       Optimal Cost: 1401      	 Gap: 2.498%
Problem: A-n36-k5        Cost: 812        Optimal Cost: 799       	 Gap: 1.627%
Problem: A-n32-k5        Cost: 843        Optimal Cost: 784       	 Gap: 7.526%
Problem: A-n62-k8        Cost: 1318       Optimal Cost: 1288      	 Gap: 2.329%
Problem: A-n53-k7        Cost: 1047       Optimal Cost: 1010      	 Gap: 3.663%
Problem: A-n63-k10       Cost: 1336       Optimal Cost: 1314      	 Gap: 1.674%
Problem: A-n60-k9        Cost: 1374       Optimal Cost: 1354      	 Gap: 1.477%
Problem: A-n65-k9        Cost: 1207       Optimal Cost: 1174      	 Gap: 2.811%
Problem: A-n69-k9        Cost: 1186     

In [11]:
# Save the results with pkl
import pickle
import os 

SAVEDIR = "results/cvrplib/"
os.makedirs(SAVEDIR, exist_ok=True)

# TODO: change the filename
with open(SAVEDIR+f'{result_name}.pkl', 'wb') as f:
    pickle.dump(results, f)

### Load data

In [12]:
import pickle

# Load
with open(SAVEDIR+f'{result_name}.pkl', 'rb') as f:
    results = pickle.load(f)

# sort results by name
results = sorted(results, key=lambda x: x["instance"])

for instance in results:
    print(f'Problem: {instance["instance"]:<15} Cost: {instance["cost"]:<10} Optimal Cost: {instance["optimal_cost"]:<10}\t Gap: {instance["gap"]:.3%}')

Problem: A-n32-k5        Cost: 843        Optimal Cost: 784       	 Gap: 7.526%
Problem: A-n33-k5        Cost: 689        Optimal Cost: 661       	 Gap: 4.236%
Problem: A-n33-k6        Cost: 751        Optimal Cost: 742       	 Gap: 1.213%
Problem: A-n34-k5        Cost: 788        Optimal Cost: 778       	 Gap: 1.285%
Problem: A-n36-k5        Cost: 812        Optimal Cost: 799       	 Gap: 1.627%
Problem: A-n37-k5        Cost: 707        Optimal Cost: 669       	 Gap: 5.680%
Problem: A-n37-k6        Cost: 982        Optimal Cost: 949       	 Gap: 3.477%
Problem: A-n38-k5        Cost: 751        Optimal Cost: 730       	 Gap: 2.877%
Problem: A-n39-k5        Cost: 835        Optimal Cost: 822       	 Gap: 1.582%
Problem: A-n39-k6        Cost: 850        Optimal Cost: 831       	 Gap: 2.286%
Problem: A-n44-k6        Cost: 973        Optimal Cost: 937       	 Gap: 3.842%
Problem: A-n45-k6        Cost: 985        Optimal Cost: 944       	 Gap: 4.343%
Problem: A-n45-k7        Cost: 1160     

In [13]:
# Take the results. Measure gap depending on first letter of the instance name
gaps_names = {}
for instance in results:
    name = instance['instance'][0]
    # if name == 'X':
    #     num_locs = instance['instance'].split('-')[1][1:]
    #     if int(num_locs) < 252:
    #         name = 'X<251'
    #     elif int(num_locs) >= 252 and int(num_locs) <= 501:
    #         name = 'X251-501'
    #     else:
    #         name = 'X501-1000'
    if name not in gaps_names:
        # if X, then we divide between < 251 and >= 251
        gaps_names[name] = []
    gaps_names[name].append(instance['gap'])
        
# Calculate the average gap for each group
average_gaps = {name: sum(gaps) / len(gaps) for name, gaps in gaps_names.items()}
for name, gap in average_gaps.items():
    print(f'Group {name}: {(gap):.3%} %')

Group A: 2.763% %
Group B: 2.886% %
Group F: 10.683% %
Group P: 3.562% %
Group X: 8.263% %


In [14]:
# Take the results. Measure gap depending on first letter of the instance name
gaps_names = {}
for instance in results:
    name = instance['instance'][0]
    if name == 'X':
        num_locs = instance['instance'].split('-')[1][1:]
        if int(num_locs) < 252:
            name = 'X<251'
        elif int(num_locs) >= 252 and int(num_locs) <= 501:
            name = 'X251-501'
        else:
            name = 'X501-1000'
    if name not in gaps_names:
        # if X, then we divide between < 251 and >= 251
        gaps_names[name] = []
    gaps_names[name].append(instance['gap'])
        
# Calculate the average gap for each group
average_gaps = {name: sum(gaps) / len(gaps) for name, gaps in gaps_names.items()}
for name, gap in average_gaps.items():
    print(f'Group {name}: {(gap):.3%}')

Group A: 2.763%
Group B: 2.886%
Group F: 10.683%
Group P: 3.562%
Group X501-1000: 12.062%
Group X<251: 4.911%
Group X251-501: 7.952%
